# Case Study: Brazilian E-commerce Customer Satisfaction Analysis
## Business Question: What drives low customer review scores at Olist and where should the business focus on?

**Author:** Adrian 

**Date:** March 2026

**Goal:** Identify which operational and product factors are most associated with low review scores (1-2 stars)

In [ ]:
# Importing Datasets
import pandas as pd
orders = pd.read_csv("/Users/adrianmar/Desktop/Personal/brazil-ecommerce/data/olist_orders_dataset.csv")
reviews = pd.read_csv("/Users/adrianmar/Desktop/Personal/brazil-ecommerce/data/olist_order_reviews_dataset.csv")
items = pd.read_csv("/Users/adrianmar/Desktop/Personal/brazil-ecommerce/data/olist_order_items_dataset.csv")
products = pd.read_csv("/Users/adrianmar/Desktop/Personal/brazil-ecommerce/data/olist_products_dataset.csv")
sellers = pd.read_csv("/Users/adrianmar/Desktop/Personal/brazil-ecommerce/data/olist_sellers_dataset.csv")
translation = pd.read_csv("/Users/adrianmar/Desktop/Personal/brazil-ecommerce/data/product_category_name_translation.csv")

In [ ]:
# Inspect dataframes to compare
dfs = {
    'orders': orders,
    'reviews': reviews,
    'items': items,
    'products': products,
    'sellers': sellers,
}

for name, df in dfs.items():
    print(f'\n{"="*50}')
    print(f'  {name.upper()}')
    print(f'{"="*50}')
    print(f'Shape: {df.shape}')
    print(f'\nHead:')
    display(df.head())
    print(f'\nInfo:')
    df.info()
    print(f'\nUnique values per column:')
    print(df.nunique())

## Info from datasets 

**orders:** 99,441 rows 

**reviews:** 99,224 rows 

**order_items:** 112,650 rows 

**products:** 32,951 rows

**sellers:** 3,095 rows


**order_items** has more rows than **orders** which shows that some orders have multiple items and if we join to order_items, it will increase row count

In [ ]:
# Join tables to create one dataset we can analyze
olist_df = orders.copy()
olist_df.shape

In [ ]:
olist_df = olist_df.merge(reviews, how="left", on="order_id") # join on left since some orders may not have reviews and do not want to drop them
olist_df.shape

In [ ]:
olist_df = olist_df.merge(items, how= "left", on = "order_id") # dataset is item-level now -> each row is one item in an order
olist_df.shape

In [ ]:
olist_df = olist_df.merge(products, how="left", on="product_id")
olist_df.shape

In [ ]:
olist_df = olist_df.merge(sellers, how="left", on="seller_id")
olist_df = olist_df.merge(translation, how ="left", on="product_category_name") # english translation names for categories
olist_df.shape

In [ ]:
# sanity checks before analysis
olist_df.head()
olist_df.info()
olist_df.isnull().sum()

In [ ]:
olist_df.to_csv("/Users/adrianmar/Desktop/Personal/brazil-ecommerce/output/olist_master_dataset.csv", index="False")

**Here, I built a final master dataset ready for analysis by joining multiple relational tables and carefully handled grain changes when the dataset moved from order-level data to item-level data.**

**Granularity:** After joining on order_items, one row represents one item in an order.

**Order ID:** is no longer unique because orders can have multiple items so the dataset will show the same order_id multiple times if that order had multiple items. 

**Average Review Score:** biased because orders with more items appear multiple times so we need to fix this moving forward. 

In [ ]:
# Convert data columns from strings to datetime
olist_df["order_purchase_timestamp"] = pd.to_datetime(olist_df["order_purchase_timestamp"])
olist_df["order_delivered_customer_date"] = pd.to_datetime(olist_df["order_delivered_customer_date"])
olist_df["order_estimated_delivery_date"] = pd.to_datetime(olist_df["order_estimated_delivery_date"])

In [ ]:
# Create delivery time/delay columns
olist_df["delivery_days"] = (olist_df["order_delivered_customer_date"] - olist_df["order_purchase_timestamp"]).dt.days # delivery time
olist_df["delay_days"] = (olist_df["order_delivered_customer_date"] - olist_df["order_estimated_delivery_date"]).dt.days # delay time (+ is late, - is early)
olist_df["is_late"] = olist_df["delay_days"] > 0 # late flag indicator
olist_df["order_value"] = olist_df["price"] + olist_df["freight_value"] # do more expensive orders get better or worse reviews?
olist_df["low_review"] = olist_df["review_score"].isin([1, 2]) # low reviews = 1 or 2 stars, main outcome variable

In [ ]:
# Olist sanity check after building our features
olist_df[['delivery_days', 'delay_days', 'is_late', 'order_value', 'low_review']].describe()

In [ ]:
olist_df['is_late'].mean()

In [ ]:
olist_df['low_review'].mean()

**% of late orders:** ~ 6.4%
This means most orders are on time or early, however late deliveries could still have an impact even if they aren't as frequent.

**% of low reviews:** ~ 16.4%
A pretty high percentage which shows that there is a real customer satisfaction problem. 

**Focus Question:** What is causing these low reviews?

**Graphical Models - explain trends or findings in a visual aspect**

In [ ]:
# First Check: Late Delivery vs Reviews - Do late deliveries translate to lower review scores?
olist_df.groupby('is_late')['review_score'].mean()
olist_df.groupby('is_late')['low_review'].mean()

In [ ]:
import matplotlib.pyplot as plt
olist_df.groupby('is_late')['low_review'].mean().plot(kind='bar')
plt.title('Late Delivery vs Bad Review Rate')
plt.ylabel('% of Bad Reviews')
plt.show()

Late deliveries significantly increase the likelihood of a low review, with a jump to over 60% of late orders receiving lower ratings compared to just 13% for on-time orders. This suggests that late orders are a primary factor towards customer dissatisfaction, yet other factors could be strong drivers as well. 

In [ ]:
# Second Check: Delay Severity - Do unhappy customers tend to experience more delay?
olist_df.groupby('is_late')['delay_days'].mean()

There is a clear separation in days it takes for an order to arrive, with not late orders typically arriving much earlier and late orders arriving much later. This helps reinforce our idea that delivery timing strongly impacts customer satisfaction.

In [ ]:
# Third Check: Worst-performing Categories
olist_df.groupby('product_category_name_english')['review_score'].mean().sort_values().head(10)

In [ ]:
olist_df.groupby('product_category_name_english')['review_score'].mean().sort_values().head(10).plot(kind='barh')
plt.title('Worst Performing Product Categories')
plt.xlabel('Average Review Score')
plt.show()

Certain product categories such as 'security and services' show consistently lower review scores, which could promote the idea that issues with different department categories could be the cause for lower reviews. 

In [ ]:
# Fourth Check: Seller Performance - Which sellers/regions are causing the delays?
olist_df.groupby('seller_state')['is_late'].mean().sort_values(ascending = False)

In [ ]:

seller_perf = olist_df.groupby('seller_state')['is_late'].mean().sort_values(ascending = False)
seller_perf.plot(kind='bar', color=plt.cm.Reds(seller_perf / seller_perf.max()))
plt.title('Top States with Highest Rate of Late Deliveries')
plt.ylabel('% Late')
plt.show()

Seller performance varies by region, with some states showing a much higher late delivery rate which could be an indicator of inefficiencies in their operations. 

In [ ]:
# Fifth Check: Price and Shipping Problems - Is the shipping cost a problem or are unhappy customers paying more?
olist_df.groupby('low_review')['order_value'].mean()

In [ ]:
olist_df.groupby('low_review')['freight_value'].mean()

In [ ]:
import seaborn as sns
sns.boxplot(x='low_review', y='freight_value', data=olist_df)
plt.title("Freight Cost Distribution by Review Outcome")
plt.xlabel("Low Review")
plt.ylabel("Freight Value")
plt.show()

Though not drastic, lower reviews are associated with higher shipping costs and customers paying more. This potentially reflects customers paying more to higher expectations and being disappointed, however would need more analysis to determine correlation. 

# Final Results and Key Insights
In this project, I explored key drivers of low customer reviews in the Olist e-commerce dataset from Kaggle. I focused on delivery times, product categories, seller behaviors, and pricing factors to see which had the biggest effects. 

**1. Delivery Delays are the strongest drivers for negative customer reviews**
Late deliveries were highly correlated with poor customer satisfaction and therefore, negative reviews. For the orders that arrived late, there was a significantly higher rate of negative reviews compared to the deliveries that arrived on-time or early. This suggest that delivery reliability is one of the most critical factors influencing customer experiences while shopping online. 

**2. Delays were substantial ones compared to a short delay**
Customers that experienced delays faced signifcantly higher delay times on average, reinforcing that not just being late, but being very late has a stronger negative impact on customer reviews. 

**3. Certain product categories consistently underperform**
Results showed that certain product categories like security and services had lower average reviews compared to other categories. This could be an indicator towards their product quality or performing lower than expectations and is a possible opening for future analysis. 
**4. Seller Performance differs by region**
There is some noticeable variation in delivery rates across the different regions that sellers operate out of. Some states show to have consistently higher delivery rates which could suggest that there are inefficiencies or challenges in their operations within these specific regions. 

**5. Price and Shipping Costs are not the main issue**
Though there is a slight increase in the average cost for dissatisfied customers, the difference is too small compared to the impact of delivery delays. This suggests that customers are more sensitive to service quality and product quality than the price. 

**Final Takeaway**
Based on the key factors that I examined throughout this project, delivery performance seems to be the most important factor regarding customer satisfaction in this dataset. This suggests that an improvement in reducing delays and logistics to ensure reliable fulfillment of orders would improve customer satisfaction over pricing nand product quality. However, one important thing to note is multivariate analysis. In this project, I focused on examining individual variables independently. However, customer satisfaction is often influenced by multiple factors and variables may appear insignificant on their own but become important when considered alongside others, and vice versa due to confounding factors. As a next steps, future analysis could involve building predictive models or running regression analysis to look at the relative impact of each factor while controlling for others. This would allow for a more complete understanding of what truly drives customer dissatisfaction. 